# 1.2 - Manipulation de Données avec NumPy

**Navigation** : [Index](../../README.md) | [>> 1.3 Pandas](1.3-Analyse_de_Donnees_avec_Pandas.ipynb)

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. Créer des tableaux NumPy (`ndarray`) et les inspecter (`shape`, `dtype`)
2. Expliquer pourquoi la **vectorisation** bat une boucle Python (et le mesurer)
3. Appliquer le **broadcasting** (diffusion de formes) et lire son message d'erreur
4. Indexer avec des **tranches**, l'indexation avancée et les **masques booléens**
5. Utiliser les réductions (`sum`, `mean`, `std`) avec la sémantique de `axis`
6. Rendre un tirage aléatoire **reproductible** avec `np.random.default_rng(seed)`

### Prérequis
- Python 3.10+
- Connaissance de base des listes Python
- Aucune expérience préalable en calcul scientifique requise

### Durée estimée : 60-75 minutes

***

Bienvenue dans ce notebook dédié à NumPy, la bibliothèque fondamentale pour le calcul scientifique en Python.

## Qu'est-ce que NumPy ?

NumPy (Numerical Python) est une bibliothèque qui fournit un objet de tableau multi-dimensionnel puissant, des routines pour des opérations rapides sur les tableaux, et des outils pour l'algèbre linéaire, les transformées de Fourier, et les nombres aléatoires.

C'est le socle sur lequel de nombreuses autres bibliothèques de data science (comme Pandas) sont construites.

> **Repère bibliographique.** NumPy est décrit et formalisé dans l'article de référence C. R. Harris et al., *Array programming with NumPy*, Nature, 585:357-362, 2020 (doi:10.1038/s41586-020-2649-2). Cet article présente l'objet `ndarray`, l'écosystème de tableaux N-dimensionnels et les fondations d'interopérabilité sur lesquelles reposent Pandas, SciPy, scikit-learn et la quasi-totalité de la data science Python.

## Création et inspection d'un tableau

L'objet principal de NumPy est le `ndarray` (n-dimensional array). On le crée à partir
d'une liste Python, mais il n'est pas une liste : c'est un **bloc de données contigu
et typé**. Deux attributs essentiels à lire en premier : `shape` (les dimensions) et
`dtype` (le type des éléments).

In [1]:
import numpy as np

# La version de NumPy utilisée dans ce notebook
print("Version de NumPy :", np.__version__)

# Création d'un tableau (ndarray) à partir d'une liste Python
ma_liste = [1, 2, 3, 4, 5]
mon_array = np.array(ma_liste)

print("\nTableau :", mon_array)
print("Type    :", type(mon_array))
print("shape   :", mon_array.shape)
print("dtype   :", mon_array.dtype)

Version de NumPy : 2.2.6

Tableau : [1 2 3 4 5]
Type    : <class 'numpy.ndarray'>
shape   : (5,)
dtype   : int64


### Liste Python vs ndarray : la même valeur, deux mémoires

Une liste Python est un tableau de **pointeurs** vers des objets `int` dispersés en
mémoire ; un `ndarray` stocke les valeurs **dans un seul bloc** (`nbytes` donne la
taille exacte des données). C'est ce qui rend NumPy à la fois plus compact et plus
rapide.

In [2]:
import sys

liste = [1, 2, 3, 4, 5]
arr = np.array(liste)

print("Liste Python : sys.getsizeof(liste) =", sys.getsizeof(liste),
      "octets (pointeurs uniquement, les ints vivent ailleurs)")
print("ndarray      : arr.nbytes            =", arr.nbytes,
      "octets (données contiguës, dtype", arr.dtype, ")")

Liste Python : sys.getsizeof(liste) = 104 octets (pointeurs uniquement, les ints vivent ailleurs)
ndarray      : arr.nbytes            = 40 octets (données contiguës, dtype int64 )


## Vectorisation : le POURQUOI de NumPy

Une opération **vectorisée** s'applique à un tableau **entier** en un appel, exécuté
en **C natif** via BLAS/LAPACK — sans boucle `for` et sans passer par l'interpréteur
Python pour chaque élément. C'est le déplacement sémantique clé : penser « tableau
entier » plutôt que « élément par élément ». La cellule suivante le **mesure** sur un
million d'éléments (le chiffre qui justifie tout le reste, mesure ici a ~x25).

In [3]:
import time

N = 1_000_000

def somme_boucle(n):
    total = 0
    for i in range(n):
        total += i
    return total

x = np.arange(N, dtype=np.int64)

t0 = time.perf_counter()
res_boucle = somme_boucle(N)
t_boucle = time.perf_counter() - t0

t0 = time.perf_counter()
res_vect = x.sum()
t_vect = time.perf_counter() - t0

print(f"Boucle Python  : somme = {res_boucle}, temps = {t_boucle:.4f} s")
print(f"Vectorisee     : somme = {res_vect}, temps = {t_vect:.4f} s")
print(f"Meme resultat  : {res_boucle == res_vect}")
print(f"Vitesse        : x{t_boucle / t_vect:.0f} plus rapide en vectorise")

Boucle Python  : somme = 499999500000, temps = 0.0234 s
Vectorisee     : somme = 499999500000, temps = 0.0009 s
Meme resultat  : True
Vitesse        : x25 plus rapide en vectorise


## Broadcasting : diffuser une forme

Le broadcasting permet d'appliquer une opération entre deux tableaux de formes
**différentes** en diffusant automatiquement les dimensions de taille 1 ou absentes.
Trois cas valides — puis le **cas d'erreur**, source n°1 d'erreurs silencieuses.

In [4]:
# Cas 1 : scalaire + tableau. Le scalaire est diffusé sur chaque élément.
a = np.arange(3)
print("a      =", a)
print("a + 10 =", a + 10, "   (le scalaire 10 est diffusé sur tout le tableau)")

a      = [0 1 2]
a + 10 = [10 11 12]    (le scalaire 10 est diffusé sur tout le tableau)


In [5]:
# Cas 2 : vecteur LIGNE (forme (3,)) diffusé sur chaque ligne d'une matrice (2,3).
m = np.arange(6).reshape(2, 3)
row = np.array([10, 20, 30])
print("m    =", m.tolist())
print("row  =", row)
print("m + row =\n", m + row, "\n   (forme (2,3) + (3,) -> (2,3))")
print("m + row shape =", (m + row).shape)

m    = [[0, 1, 2], [3, 4, 5]]
row  = [10 20 30]
m + row =
 [[10 21 32]
 [13 24 35]] 
   (forme (2,3) + (3,) -> (2,3))
m + row shape = (2, 3)


In [6]:
# Cas 3 : vecteur COLONNE (forme (2,1)) diffusé sur chaque colonne.
col = np.array([[100], [200]])   # forme (2,1)
print("col  =\n", col)
print("m + col =\n", m + col, "\n   (forme (2,3) + (2,1) -> (2,3))")
print("m + col shape =", (m + col).shape)

col  =
 [[100]
 [200]]
m + col =
 [[100 101 102]
 [203 204 205]] 
   (forme (2,3) + (2,1) -> (2,3))
m + col shape = (2, 3)


In [7]:
# Cas d'erreur : formes (2,3) et (2,) incompatibles. NumPy lève une ValueError
# explicite — on la capture pour en lire le message.
try:
    m + np.array([1, 2])          # (2,3) + (2,) : le 3 ne s'aligne pas avec le 2
except ValueError as e:
    print("ValueError levée :")
    print("   ", e)

print("\nInterprétation : les formes (2,3) et (2,) ne sont pas compatibles pour")
print("l'élément le plus à droite. Il faut une forme (3,) (ligne) ou (2,1)")
print("(colonne) pour que le broadcasting fonctionne.")

ValueError levée :
    operands could not be broadcast together with shapes (2,3) (2,) 

Interprétation : les formes (2,3) et (2,) ne sont pas compatibles pour
l'élément le plus à droite. Il faut une forme (3,) (ligne) ou (2,1)
(colonne) pour que le broadcasting fonctionne.


## Indexation : tranches, indexation avancée, masques booléens

Trois façons d'extraire. Les **tranches** (`a[1:4]`, `X[:, 0]`) découpent ;
l'**indexation avancée** (`a[[0, 2, 4]]`) prend une liste d'indices ; le **masque
booléen** (`a[a > x]`) filtre par une condition — l'idiome le plus utilisé en pratique.

In [8]:
a = np.array([10, 20, 30, 40, 50])
X = np.arange(12).reshape(3, 4)

print("a       =", a)
print("a[1:4]  =", a[1:4], "    (tranche : indices 1 a 3)")
print("X       =\n", X)
print("X[:, 0] =", X[:, 0], "   (colonne 0 : TOUTES les lignes, l'idiome X[:, 0])")
print("X[1:, 2:] =\n", X[1:, 2:], " (sous-bloc lignes 1-2, colonnes 2-3)")

a       = [10 20 30 40 50]
a[1:4]  = [20 30 40]     (tranche : indices 1 a 3)
X       =
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
X[:, 0] = [0 4 8]    (colonne 0 : TOUTES les lignes, l'idiome X[:, 0])
X[1:, 2:] =
 [[ 6  7]
 [10 11]]  (sous-bloc lignes 1-2, colonnes 2-3)


In [9]:
a = np.array([10, 20, 30, 40, 50])
X = np.arange(12).reshape(3, 4)

print("a[[0, 2, 4]]        =", a[[0, 2, 4]], "   (indices choisis)")
print("X[[0, 2], [1, 3]]   =", X[[0, 2], [1, 3]], "   (paires (0,1) et (2,3))")

a[[0, 2, 4]]        = [10 30 50]    (indices choisis)
X[[0, 2], [1, 3]]   = [ 1 11]    (paires (0,1) et (2,3))


In [10]:
a = np.array([5, 12, 18, 9, 25, 3])

# Le masque est lui-même un tableau de booléens, comparé d'un coup.
print("a        =", a)
print("a > 10   =", a > 10, "   (masque : True/False par élément)")
print("a[a > 10] =", a[a > 10], "   (sélection où le masque est True)")

# Masque composé : TOUJOURS une parenthèse autour de chaque comparaison avec &.
print("\n(a > 5) & (a < 20)  =", (a > 5) & (a < 20))
print("a[(a > 5) & (a < 20)] =", a[(a > 5) & (a < 20)])

a        = [ 5 12 18  9 25  3]
a > 10   = [False  True  True False  True False]    (masque : True/False par élément)
a[a > 10] = [12 18 25]    (sélection où le masque est True)

(a > 5) & (a < 20)  = [False  True  True  True False False]
a[(a > 5) & (a < 20)] = [12 18  9]


## Réductions et la sémantique de `axis`

`sum`, `mean`, `std` réduisent le tableau. Le paramètre `axis` dit **quelle dimension
on écrase** : `axis=0` réduit toutes les lignes (le résultat garde une entrée par
colonne), `axis=1` réduit toutes les colonnes (une entrée par ligne). C'est la
confusion classique — la cellule suivante rend le résultat visuel.

In [11]:
X = np.array([[1, 2, 3], [4, 5, 6]])   # forme (2,3)

print("X              :", X.shape, "\n", X)
print("X.sum()        =", X.sum(), "   (tous les éléments)")
print("X.sum(axis=0)  =", X.sum(axis=0), "  (écrase les lignes -> forme (3,))")
print("X.sum(axis=1)  =", X.sum(axis=1), "  (écrase les colonnes -> forme (2,))")
print("X.mean(axis=0) =", X.mean(axis=0))
print("X.std(axis=1)  =", X.std(axis=1))

X              : (2, 3) 
 [[1 2 3]
 [4 5 6]]
X.sum()        = 21    (tous les éléments)
X.sum(axis=0)  = [5 7 9]   (écrase les lignes -> forme (3,))
X.sum(axis=1)  = [ 6 15]   (écrase les colonnes -> forme (2,))
X.mean(axis=0) = [2.5 3.5 4.5]
X.std(axis=1)  = [0.81649658 0.81649658]


## Reproductibilité : `np.random.default_rng(seed)`

En ML, un tirage aléatoire doit être **reproductible** : même graine, mêmes tirages.
`np.random.default_rng(seed)` construit un générateur — la cellule le démontre en
refaisant deux fois le même tirage avec la même graine.

In [12]:
rng = np.random.default_rng(42)
tirage_a = rng.normal(size=5)

rng_bis = np.random.default_rng(42)
tirage_b = rng_bis.normal(size=5)

print("Tirage 1 (seed=42) :", tirage_a)
print("Tirage 2 (seed=42) :", tirage_b)
print("Identiques ?", (tirage_a == tirage_b).all())

# Sans graine fixe, le générateur est différent à chaque exécution.
rng_neutre = np.random.default_rng()
print("\nSans graine (default_rng()) :", rng_neutre.normal(size=3),
      "  <- changera au prochain run")

Tirage 1 (seed=42) : [ 0.30471708 -1.03998411  0.7504512   0.94056472 -1.95103519]
Tirage 2 (seed=42) : [ 0.30471708 -1.03998411  0.7504512   0.94056472 -1.95103519]
Identiques ? True

Sans graine (default_rng()) : [-1.24277634 -0.21680314  1.12479784]   <- changera au prochain run


## Exercices fondamentaux

### Exercice A — vectorisez une boucle

La version Python (fournie) calcule le carré de chaque élément dans une boucle.
À vous de faire le même calcul en **une ligne vectorisée**.

In [13]:
valeurs = np.array([5, 12, 8, 20, 3, 15])

# Version boucle (fournie) :
carres_boucle = np.empty_like(valeurs)
for i in range(len(valeurs)):
    carres_boucle[i] = valeurs[i] ** 2

# TODO: refaites ce calcul SANS boucle, en une ligne vectorisée.
# Indice: l'operateur ** s'applique element par element sur un ndarray.
carres_vectorises = None  # Remplacez None par l'operation vectorisee

if carres_vectorises is not None:
    print("Boucle      :", carres_boucle)
    print("Vectorisee  :", carres_vectorises)
    print("Identiques ?", (carres_boucle == carres_vectorises).all())
else:
    print("Exercice a completer : vectorisez ce calcul (carres_vectorises)")

Exercice a completer : vectorisez ce calcul (carres_vectorises)


### Exercice B — filtrez avec un masque composé

Filtrez les valeurs **strictement comprises entre 5 et 20 (exclus)** en une ligne,
avec des parenthèses autour de chaque comparaison.

In [14]:
donnees = np.array([3, 15, 7, 22, 11, 30, 4, 18])

# TODO: selectionnez les valeurs strictement entre 5 et 20 (exclus).
# Indice: (donnees > 5) & (donnees < 20) — parentheses obligatoires avec &
selection = None  # Remplacez None

if selection is not None:
    print("donnees    :", donnees)
    print("selection  :", selection)
else:
    print("Exercice a completer : filtrez entre 5 et 20 (exclus)")

Exercice a completer : filtrez entre 5 et 20 (exclus)

### Exercice C — appliquez un broadcasting

Un tableau de notes (3 étudiants x 3 matières) : ajoutez un **bonus de 2 points** à
chaque note, en une seule opération de broadcasting.

In [15]:
notes = np.array([
    [12, 15, 9],
    [8, 14, 17],
    [16, 11, 13],
])   # forme (3,3) : 3 etudiants, 3 matieres

# TODO: ajoutez 2 points a chaque note via broadcasting (notes + scalaire).
bonus = 2
notes_bonus = None  # Remplacez None : notes + bonus

if notes_bonus is not None:
    print("notes       :\n", notes)
    print("notes_bonus :\n", notes_bonus)
else:
    print("Exercice a completer : ajoutez un bonus de 2 points via broadcasting")

Exercice a completer : ajoutez un bonus de 2 points via broadcasting


## Exercices avancés

## Exercice 2

Créez un tableau NumPy de 10 éléments allant de 0 a 9, puis calculez :
1. La somme de tous les éléments
2. La moyenne des éléments
3. Le carre de chaque élément

Indices :
-  pour créer le tableau
- ,  pour les statistiques
- NumPy supporte les opérations élément par élément (ex: )


In [16]:
# Exercice : Multipliez le tableau par 2 et affichez le resultat
array_exercice = np.array([2, 4, 6, 8, 10])

# TODO: Multipliez chaque element de array_exercice par 2
# Indice: NumPy permet les operations vectorisees (ex: array * scalaire)
resultat = None  # Remplacez None par l'operation appropriee

print(f"Tableau original : {array_exercice}")
print(f"Tableau multiplie par 2 : {resultat}")


Tableau original : [ 2  4  6  8 10]
Tableau multiplie par 2 : None


## Exercice : Statistiques sur un Dataset

À vous de pratiquer les opérations vectorisées avec NumPy !

### Objectifs
Créez un tableau de données synthétiques et calculez des statistiques descriptives.

### Instructions



In [17]:
import numpy as np

# TODO: Créez un tableau de 100 valeurs aléatoires entre 0 et 100
# Indice: utilisez np.random.randint()
donnees = None  # Remplacez None

# TODO: Calculez les statistiques suivantes
somme = None      # Utilisez np.sum()
moyenne = None    # Utilisez np.mean()
minimum = None    # Utilisez np.min()
maximum = None    # Utilisez np.max()
ecart_type = None # Utilisez np.std()

# TODO: Créez un masque booléen pour les valeurs > 50
masque = None     # données > 50
valeurs_sup_50 = None  # Appliquez le masque

# Affichage des résultats
if donnees is not None and valeurs_sup_50 is not None:
    print(f"Somme: {somme}")
    print(f"Moyenne: {moyenne}")
    print(f"Min: {minimum}, Max: {maximum}")
    print(f"Écart-type: {ecart_type}")
    print(f"Valeurs > 50: {len(valeurs_sup_50)} sur {len(donnees)}")
else:
    print("Exercice à compléter : remplacez les None par votre code")


Exercice à compléter : remplacez les None par votre code


### Exercice 3 : Opérations sur les matrices 2D

NumPy excelle dans les opérations sur les tableaux multidimensionnels. Vous allez créer une matrice 2D (tableau de tableaux) et appliquer des opérations avancees : transposition, produit matriciel et extraction de sous-matrices.

**Objectif** : Manipuler une matrice 3x3 avec les fonctions de NumPy pour comprendre les bases de l'algebre lineaire.

**Indices** :
- Utilisez `np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])` pour créer une matrice 3x3
- `matrice.T` ou `np.transpose(matrice)` pour la transposition
- `np.dot(A, B)` ou l'opérateur `@` pour le produit matriciel
- `matrice[0:2, 1:3]` pour extraire une sous-matrice (slicing)

In [18]:
# Exercice 3 : Operations sur les matrices 2D
# Creez et manipulez une matrice avec NumPy

# Etape 1: Creez une matrice 3x3
# Indice: np.array([[1,2,3], [4,5,6], [7,8,9]])
matrice = None  # Remplacez None

# Etape 2: Calculez la transposee
# Indice: matrice.T
transposee = None  # Remplacez None

# Etape 3: Calculez le produit matriciel de la matrice par elle-meme
# Indice: np.dot(matrice, matrice) ou matrice @ matrice
produit = None  # Remplacez None

# Etape 4: Extrayez la sous-matrice 2x2 en haut a gauche
# Indice: matrice[0:2, 0:2]
sous_matrice = None  # Remplacez None

# Affichage
if matrice is not None:
    print(f"Matrice originale :\n{matrice}")
    print(f"\nTransposee :\n{transposee}")
    print(f"\nProduit matriciel :\n{produit}")
    print(f"\nSous-matrice 2x2 :\n{sous_matrice}")
else:
    print("Exercice 3 a completer : operations sur les matrices 2D")

Exercice 3 a completer : operations sur les matrices 2D


## Conclusion

Ce notebook a posé les **fondations NumPy** indispensables à toute la data science
Python — le socle sur lequel s'appuient Pandas (notebook [1.3](1.3-Analyse_de_Donnees_avec_Pandas.ipynb)),
scikit-learn et les frameworks de deep learning.

**Ce qu'il faut retenir** :

- **L'objet `ndarray`** est un tableau N-dimensionnel **typé et contigu en mémoire** :
  `shape` et `dtype` se lisent avant toute opération, et le stockage compact
  (`nbytes`) explique pourquoi NumPy bat les listes Python en vitesse et en mémoire.
- **La vectorisation** remplace les boucles `for` : `a + b`, `a * 2`, `a.sum()`
  s'appliquent élément par élément via du code C natif — mesuré ici à un facteur
  **~x25** sur un million d'éléments (dépend de la machine). C'est le déplacement sémantique clé.
- **Le broadcasting** diffuse une forme sur une autre quand les dimensions sont
  compatibles (scalaire, ligne, colonne) ; le **message d'erreur** de NumPy
  (ValueError) est fait pour être lu, pas deviné.
- **L'indexation** — tranches (`X[:, 0]`), indexation avancée, **masques booléens**
  (`a[a > x]`, `(a > 5) & (a < 20)`) — est l'idiome le plus courant : lire
  `X[:, 0]` et `X[X > seuil]` est le prérequis de toute la suite.
- **`axis`** dit quelle dimension une réduction écrase : `axis=0` les lignes,
  `axis=1` les colonnes.
- **`np.random.default_rng(seed)`** rend un tirage reproductible — même graine,
  mêmes tirages — indispensable en ML.

**Le pont vers Pandas** : NumPy manipule des tableaux numériques nus ; Pandas (1.3)
ajoute l'indexation par étiquettes et le typage hétérogène — mais tout `DataFrame`
Pandas enveloppe un `ndarray`. Maîtriser la vectorisation, le broadcasting et les
masques ici, c'est comprendre pourquoi une opération Pandas vectorisée bat toujours
une boucle `iterrows`.

**Pour aller plus loin** : les fonctions d'algèbre linéaire de `np.linalg` (inverse,
déterminant, valeurs propres) et l'indexation par tableau d'indices 2D.
Les exercices avancés ci-dessus (transposition, produit matriciel, sous-matrices)
servent de point d'entrée vers `np.linalg`.

## Références

1. C. R. Harris et al., *Array programming with NumPy*, Nature, 585(7825):357-362, 2020. doi:10.1038/s41586-020-2649-2. Article de référence décrivant l'objet `ndarray`, l'écosystème de tableaux N-dimensionnels et les fondations de la data science Python.
2. S. van der Walt, S. C. Colbert, G. Varoquaux, *The NumPy Array: A Structure for Efficient Numerical Computation*, Computing in Science & Engineering, 13(2):22-30, 2011. Architecture interne de `ndarray` et principes de vectorisation.
3. T. E. Oliphant, *A Guide to NumPy*, Trelgol Publishing, 2006. Référence technique historique sur NumPy (première édition).